# AWS Athena Tools

This notebook demonstrates the programmatic usage of the two AWS Athena helper tools:
1. **`aws_athena_table_search`** — Search for a table across all Glue/Athena databases
2. **`aws_athena_cleanup`** — Identify and optionally drop stale tables/views

Both tools are also available as CLI commands after installing the package.

In [ ]:
from buildstock_query.tools import aws_athena_table_search, aws_athena_cleanup

## 1. Search for a Table Across All Databases

Use `aws_athena_table_search` to find which databases contain a table matching a given name.
Supports exact match, substring match, and regex.

In [ ]:
# Exact match: find databases that contain a table named "baseline"
matches = aws_athena_table_search(
    table_name="baseline",
    region="us-west-2",
)
matches

In [ ]:
# Substring match: find tables whose name contains "timeseries"
matches = aws_athena_table_search(
    table_name="timeseries",
    region="us-west-2",
    substring=True,
)
matches

In [ ]:
# Regex match: find tables matching a pattern
matches = aws_athena_table_search(
    table_name=r"baseline_\d+",
    region="us-west-2",
    regex=True,
)
matches

In [ ]:
# Filter to only search databases whose name contains "resstock"
matches = aws_athena_table_search(
    table_name="baseline",
    region="us-west-2",
    database_filter="resstock",
)
matches

## 2. Cleanup Stale Athena Tables and Views

Use `aws_athena_cleanup` to scan a database and find tables/views whose underlying
S3 data no longer exists. By default it only **reports** stale objects; pass `drop=True`
to actually remove them.

In [ ]:
# Dry-run: identify stale tables without dropping
summary = aws_athena_cleanup(
    database="my_db",
    workgroup="primary",
    region="us-west-2",
    drop=False,
)
summary

In [ ]:
# Inspect results
print(f"Stale tables: {summary['stale_tables']}")
print(f"Stale views:  {summary['stale_views']}")
print(f"Healthy tables: {len(summary['healthy_tables'])}")
print(f"Healthy views:  {len(summary['healthy_views'])}")

In [ ]:
# Skip view inspection (tables only)
summary = aws_athena_cleanup(
    database="my_db",
    workgroup="primary",
    region="us-west-2",
    drop=False,
    skip_views=True,
)
summary

In [ ]:
# Actually drop stale objects (use with caution!)
# summary = aws_athena_cleanup(
#     database="my_db",
#     workgroup="primary",
#     region="us-west-2",
#     drop=True,
# )

## CLI Equivalents

These tools are also available as command-line scripts:

```bash
# Search for a table
aws_athena_table_search -t baseline -s -f resstock

# Cleanup stale tables (dry-run)
aws_athena_cleanup -d my_db -w primary

# Cleanup stale tables (actually drop)
aws_athena_cleanup -d my_db -w primary -D
```